# Student A — Data Preparation Pipeline

In this notebook, we document the data preparation work for our phishing URL detection project.

We use this notebook to:
- inspect the raw dataset,
- confirm that it contains raw URL strings and binary labels,
- check missing values,
- check duplicate URLs,
- check conflicting labels,
- clean the dataset.

This step is important because the quality of the dataset directly affects the reliability of our models. In particular, duplicated URLs must be handled before splitting to avoid data leakage between the training and test sets.


## Dataset Source

We use the Mendeley 2026 phishing URL dataset:

**Phishing URL dataset**  
DOI: `10.17632/3jddhy2f6s.1`

The dataset contains raw URLs with binary labels:
- `0` = legitimate URL
- `1` = phishing URL

This dataset fits our project because it provides raw URL strings, which we need for both handcrafted feature extraction and character-level modeling.


## Import Libraries

In [ ]:
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

import pandas as pd


## Configuration


In [4]:
data_path = Path("../data/raw/mendeley_2026_commoncrawl_phishtank.csv")

## Load Raw Data

We load the dataset as it is from the raw data folder.


In [5]:
df = pd.read_csv(data_path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Shape: (149726, 2)
Columns: ['URL', 'Label']


,URL,Label
0,https://optus-myaccount.s2-tastewp.com/,1
1,https://www.googleapis.com/auth/drive.file';,0
2,https://wikipedia.org/,0
3,https://www.aol.com/2008-01-23-ask-the-dolans-...,0
4,https://rebrand.ly/uda4njv,1


## Quick Data Check

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 149726 entries, 0 to 149725
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   URL     149726 non-null  str  
 1   Label   149726 non-null  int64
dtypes: int64(1), str(1)
memory usage: 12.6 MB


In [7]:
df.sample(5, random_state=42)

,URL,Label
125133,https://qcaprod.australiaeast.cloudapp.azure.c...,0
15585,https://usps.bbntnczlrw.top/,1
25447,http://www.theguardian.com/robots.txt,0
143062,https://pub-9ec1967945504971b7ee3cd05fc7a4f9.r...,1
48501,https://ipfs.eth.aragon.network/ipfs/bafkreidd...,1


The dataset has raw URL strings and numeric binary labels, which fits our project setup.

## Missing Values

In [10]:
df.isna().sum()

URL      0
Label    0
dtype: int64

No missing values were found, so no rows need to be removed for missing data.

## Label Distribution

In [13]:
df["Label"].value_counts().sort_index()

Label
0    94919
1    54807
Name: count, dtype: int64

In [12]:
df["Label"].value_counts(normalize=True).sort_index().round(4)

Label
0    0.634
1    0.366
Name: proportion, dtype: float64

The dataset is imbalanced, but it still has both classes. Around 63.4% of the URLs are legitimate and 36.6% are phishing.

Later, we should use stratified splitting so the train, validation, and test sets keep similar class proportions.


## Duplicate Check

In [16]:
# We check duplicate URLs before splitting to avoid the same URL appearing in multiple sets.
df["URL"].duplicated().sum()

np.int64(19949)

There are 19,949 duplicate URL rows. We should remove duplicates before splitting so the same URL does not appear in both training and test data.

## Conflicting Label Check


In [18]:
# Check if the same URL has more than one label.
df.groupby("URL")["Label"].nunique().gt(1).sum()

np.int64(0)

No conflicting labels were found. This means the same URL is not labeled as both legitimate and phishing.

## Cleaning Duplicate URLs

In [19]:
# Remove duplicate URLs before splitting.
df_clean = df.drop_duplicates(subset="URL").copy()

print("Original shape:", df.shape)
print("Clean shape:", df_clean.shape)
print("Removed rows:", len(df) - len(df_clean))

Original shape: (149726, 2)
Clean shape: (129777, 2)
Removed rows: 19949


After removing duplicate URLs, the dataset now has 129,777 unique URL records.

## Label Distribution After Cleaning

In [20]:
# Check class counts after removing duplicate URLs.
df_clean["Label"].value_counts().sort_index()

Label
0    74972
1    54805
Name: count, dtype: int64

In [21]:
# Check class ratios after removing duplicate URLs.
df_clean["Label"].value_counts(normalize=True).sort_index().round(4)

Label
0    0.5777
1    0.4223
Name: proportion, dtype: float64

After duplicate removal, the cleaned dataset still contains both classes. The class ratio changed because many duplicate rows were from the legitimate class.

## Save Clean Dataset

In [22]:
# Save the cleaned dataset for the next notebooks.
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

clean_path = processed_dir / "clean_urls.csv"
df_clean.to_csv(clean_path, index=False)

print("Saved to:", clean_path)
print("Final shape:", df_clean.shape)

Saved to: ..\data\processed\clean_urls.csv
Final shape: (129777, 2)


The cleaned dataset was saved successfully and will be used in the next step for train/validation/test splitting.

## Dataset Audit Summary


In [23]:
# Final numbers for the data audit.
print("Raw rows:", len(df))
print("Clean rows:", len(df_clean))
print("Removed duplicate rows:", len(df) - len(df_clean))

print("\nClean label counts:")
print(df_clean["Label"].value_counts().sort_index())

print("\nClean label ratios:")
print(df_clean["Label"].value_counts(normalize=True).sort_index().round(4))

Raw rows: 149726
Clean rows: 129777
Removed duplicate rows: 19949

Clean label counts:
Label
0    74972
1    54805
Name: count, dtype: int64

Clean label ratios:
Label
0    0.5777
1    0.4223
Name: proportion, dtype: float64


The raw dataset contained 149,726 rows. After checking duplicates and conflicting labels, we removed 19,949 duplicate URL rows and kept 129,777 unique URLs.

The cleaned dataset contains 74,972 legitimate URLs and 54,805 phishing URLs. This gives us a usable binary dataset for the next stage.

Next, we will use this cleaned dataset for stratified train/validation/test splitting and feature engineering.
